# 🔢 NumPy — From Basics to Advanced

NumPy (**Num**erical **Py**thon) is the foundation of the Python scientific/data stack —
Pandas, Matplotlib, scikit-learn, PyTorch, etc. are all built on top of it (or interoperate
with it). Its core object is the **ndarray** — an N-dimensional array that is much faster and
more memory-efficient than a plain Python list, because it stores data in contiguous memory
and operates on it using vectorized (compiled C) operations instead of slow Python loops.

## Table of Contents
1. [Arrays: creation & basics](#1)
2. [Indexing, slicing, boolean masking](#2)
3. [Array shapes & reshaping](#3)
4. [Vectorized math & broadcasting](#4)
5. [Aggregations & statistics](#5)
6. [Random numbers](#6)
7. [Linear algebra](#7)
8. [Advanced: views vs copies, stacking, performance](#8)

In [1]:
import numpy as np
print(np.__version__)


2.4.6


<a id="1"></a>
# 1. Arrays: creation & basics


In [2]:
# =========================================================
# CREATING ARRAYS
# =========================================================

a = np.array([1, 2, 3, 4, 5])              # 1D array, from a Python list
b = np.array([[1, 2, 3], [4, 5, 6]])       # 2D array, from a list of lists

print(a)
print(b)

print(a.ndim, a.shape, a.dtype, a.size)    # dimensions, shape, data type, total element count
print(b.ndim, b.shape, b.dtype, b.size)

# Common array constructors - much faster than building lists then converting
zeros = np.zeros((2, 3))          # 2x3 array of 0.0
ones = np.ones((3,))              # 1D array of 1.0
full = np.full((2, 2), 7)         # array filled with a constant
identity = np.eye(3)              # 3x3 identity matrix
arange = np.arange(0, 10, 2)      # like range(), but returns an array -> [0,2,4,6,8]
linspace = np.linspace(0, 1, 5)   # 5 evenly spaced numbers from 0 to 1 (inclusive)

for name, arr in [("zeros", zeros), ("ones", ones), ("full", full),
                   ("identity", identity), ("arange", arange), ("linspace", linspace)]:
    print(name, "->", arr)

[1 2 3 4 5]
[[1 2 3]
 [4 5 6]]
1 (5,) int64 5
2 (2, 3) int64 6
zeros -> [[0. 0. 0.]
 [0. 0. 0.]]
ones -> [1. 1. 1.]
full -> [[7 7]
 [7 7]]
identity -> [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
arange -> [0 2 4 6 8]
linspace -> [0.   0.25 0.5  0.75 1.  ]


In [3]:
# =========================================================
# DTYPES - all elements in a numpy array share ONE data type
# This is what makes numpy fast (unlike Python lists, which can mix types)
# =========================================================

int_arr = np.array([1, 2, 3], dtype=np.int32)
float_arr = np.array([1, 2, 3], dtype=np.float64)
bool_arr = np.array([1, 0, 1], dtype=bool)

print(int_arr.dtype, float_arr.dtype, bool_arr.dtype)

# Casting an existing array to a different dtype
casted = int_arr.astype(np.float64)
print(casted, casted.dtype)

int32 float64 bool
[1. 2. 3.] float64


<a id="2"></a>
# 2. Indexing, slicing, boolean masking


In [4]:
# =========================================================
# INDEXING & SLICING - similar to Python lists, but extends to N dimensions
# =========================================================

arr = np.array([10, 20, 30, 40, 50])
print(arr[0], arr[-1])          # single element access
print(arr[1:4])                 # slice -> [20, 30, 40]
print(arr[::-1])                # reversed

matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(matrix[1, 2])              # row 1, col 2 -> 6  (comma syntax, unlike nested lists)
print(matrix[0])                 # entire first row -> [1, 2, 3]
print(matrix[:, 1])              # entire second COLUMN -> [2, 5, 8]
print(matrix[0:2, 0:2])          # top-left 2x2 sub-matrix

10 50
[20 30 40]
[50 40 30 20 10]
6
[1 2 3]
[2 5 8]
[[1 2]
 [4 5]]


In [5]:
# =========================================================
# BOOLEAN MASKING - filtering data by condition (extremely common in real use)
# =========================================================

arr = np.array([1, -2, 3, -4, 5, -6])

mask = arr > 0                    # array of True/False, same shape as arr
print(mask)
print(arr[mask])                  # only the elements where mask is True -> [1, 3, 5]
print(arr[arr > 0])               # same thing, written inline (very common pattern)

arr[arr < 0] = 0                  # conditional assignment: replace all negatives with 0
print(arr)

# Fancy indexing - select elements using a list/array of indices
idx = [0, 2, 4]
print(np.array([10, 20, 30, 40, 50])[idx])   # -> [10, 30, 50]

[ True False  True False  True False]
[1 3 5]
[1 3 5]
[1 0 3 0 5 0]
[10 30 50]


In [6]:
# =========================================================
# RESHAPING
# =========================================================

arr = np.arange(12)              # [0, 1, ..., 11]
print(arr)

reshaped = arr.reshape(3, 4)     # reorganize into 3 rows x 4 cols (same data, new shape)
print(reshaped)

print(reshaped.reshape(-1))      # -1 means "flatten" / infer automatically -> back to 1D
print(reshaped.flatten())        # equivalent, always returns a COPY
print(reshaped.T)                # transpose - flips rows and columns
print(reshaped.reshape(4, -1))   # -1 in one dimension means "figure this size out for me"


[ 0  1  2  3  4  5  6  7  8  9 10 11]
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[ 0  1  2  3  4  5  6  7  8  9 10 11]
[[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


In [7]:
# =========================================================
# VECTORIZED OPERATIONS - apply math to WHOLE arrays at once, no manual loops
# This is the #1 reason numpy is fast: operations run in compiled C, not Python.
# =========================================================

a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print(a + b)          # element-wise addition -> [11, 22, 33, 44]
print(a * b)          # element-wise multiplication
print(a ** 2)          # element-wise power
print(a > 2)           # element-wise comparison -> boolean array

# Universal functions (ufuncs) - vectorized math functions
print(np.sqrt(a))
print(np.exp(a))
print(np.log(a))
print(np.sin(a))


[11 22 33 44]
[ 10  40  90 160]
[ 1  4  9 16]
[False False  True  True]
[1.         1.41421356 1.73205081 2.        ]
[ 2.71828183  7.3890561  20.08553692 54.59815003]
[0.         0.69314718 1.09861229 1.38629436]
[ 0.84147098  0.90929743  0.14112001 -0.7568025 ]


In [8]:
# =========================================================
# BROADCASTING - numpy's rules for combining arrays of DIFFERENT shapes
# The smaller array is "stretched" (conceptually, not literally in memory) to match.
# =========================================================

matrix = np.array([[1, 2, 3], [4, 5, 6]])    # shape (2, 3)
row = np.array([10, 20, 30])                  # shape (3,)

print(matrix + row)   # row is broadcast across every row of the matrix

scalar = 100
print(matrix + scalar)   # scalar is broadcast to every element

# Column vector broadcasting - needs an explicit extra dimension
col = np.array([[1], [2]])       # shape (2, 1)
print(matrix + col)              # broadcasts across every column

[[11 22 33]
 [14 25 36]]
[[101 102 103]
 [104 105 106]]
[[2 3 4]
 [6 7 8]]


In [9]:
# =========================================================
# AGGREGATIONS
# =========================================================

arr = np.array([[1, 2, 3], [4, 5, 6]])

print(arr.sum())            # sum of ALL elements -> 21
print(arr.sum(axis=0))      # sum DOWN each column -> [5, 7, 9]
print(arr.sum(axis=1))      # sum ACROSS each row   -> [6, 15]

print(arr.mean(), arr.std(), arr.var())   # mean, standard deviation, variance
print(arr.min(), arr.max())
print(arr.argmin(), arr.argmax())         # INDEX of the min/max (flattened index)

print(np.median(arr))
print(np.percentile(arr, 90))             # 90th percentile

21
[5 7 9]
[ 6 15]
3.5 1.707825127659933 2.9166666666666665
1 6
0 5
3.5
5.5


In [10]:
# =========================================================
# RANDOM NUMBER GENERATION
# =========================================================

rng = np.random.default_rng(seed=42)   # modern, recommended API - seed makes it REPRODUCIBLE

print(rng.random(5))                    # 5 random floats in [0, 1)
print(rng.integers(1, 100, size=5))     # 5 random ints in [1, 100)
print(rng.normal(loc=0, scale=1, size=5))   # samples from a normal (Gaussian) distribution

arr = np.arange(10)
rng.shuffle(arr)             # shuffles in-place
print(arr)

print(rng.choice([1, 2, 3, 4, 5], size=3, replace=False))   # random sample without replacement

[0.77395605 0.43887844 0.85859792 0.69736803 0.09417735]
[53 97 73 76 72]
[-0.01680116 -0.85304393  0.87939797  0.77779194  0.0660307 ]
[0 1 8 4 7 2 9 5 6 3]
[1 3 5]


In [11]:
# =========================================================
# LINEAR ALGEBRA
# =========================================================

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print(A @ B)                    # matrix multiplication (preferred way, via @ operator)
print(np.dot(A, B))             # equivalent to A @ B for 2D arrays

print(np.linalg.det(A))         # determinant
print(np.linalg.inv(A))         # matrix inverse
print(np.linalg.eig(A))         # eigenvalues & eigenvectors

v = np.array([1, 2])
print(A @ v)                    # matrix-vector product
print(np.linalg.norm(v))        # vector magnitude (Euclidean norm)

[[19 22]
 [43 50]]
[[19 22]
 [43 50]]
-2.0000000000000004
[[-2.   1. ]
 [ 1.5 -0.5]]
EigResult(eigenvalues=array([-0.37228132,  5.37228132]), eigenvectors=array([[-0.82456484, -0.41597356],
       [ 0.56576746, -0.90937671]]))
[ 5 11]
2.23606797749979


In [12]:
# =========================================================
# VIEWS vs COPIES - a common source of subtle bugs
# =========================================================

arr = np.array([1, 2, 3, 4, 5])
view = arr[1:4]          # SLICING returns a VIEW - shares memory with the original!
view[0] = 999
print(arr)                # original IS changed -> [1, 999, 3, 4, 5]

arr2 = np.array([1, 2, 3, 4, 5])
copy = arr2[1:4].copy()   # .copy() forces an independent copy
copy[0] = 999
print(arr2)                # original is UNCHANGED

print(np.shares_memory(arr, view))    # True
print(np.shares_memory(arr2, copy))   # False

[  1 999   3   4   5]
[1 2 3 4 5]
True
False


In [13]:
# =========================================================
# STACKING & SPLITTING ARRAYS
# =========================================================

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print(np.vstack([a, b]))          # stack vertically -> 2x3 array (as new rows)
print(np.hstack([a, b]))          # stack horizontally -> concatenate into 1D
print(np.concatenate([a, b]))     # general-purpose concatenation

matrix = np.arange(9).reshape(3, 3)
print(np.split(matrix, 3))        # split into 3 equal pieces along axis 0

[[1 2 3]
 [4 5 6]]
[1 2 3 4 5 6]
[1 2 3 4 5 6]
[array([[0, 1, 2]]), array([[3, 4, 5]]), array([[6, 7, 8]])]


In [14]:
# =========================================================
# WHY VECTORIZATION MATTERS - a quick performance comparison
# =========================================================
import time

n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

start = time.time()
py_result = [x * 2 for x in py_list]         # pure Python loop (via comprehension)
py_time = time.time() - start

start = time.time()
np_result = np_arr * 2                        # vectorized numpy operation
np_time = time.time() - start

print(f"Python list comprehension: {py_time:.4f}s")
print(f"NumPy vectorized op:       {np_time:.4f}s")
print(f"NumPy was ~{py_time / np_time:.0f}x faster")

Python list comprehension: 0.0696s
NumPy vectorized op:       0.0019s
NumPy was ~37x faster
